In [1]:
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder



In [2]:
BASE_DIR = Path.cwd().parent

STAGE2_TRAIN_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage2"
    / "stage2_train_df.csv"
)
STAGE2_VAL_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "stage2"
    / "stage2_val_df.csv"
)

print("Current directory:", Path.cwd())
print("Stage 2 train file:", STAGE2_TRAIN_PATH.resolve())
print("Stage 2 val file:", STAGE2_VAL_PATH.resolve())

print("File exists:", STAGE2_TRAIN_PATH.exists())
print("File exists:", STAGE2_VAL_PATH.exists())

Current directory: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\notebooks
Stage 2 train file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_train_df.csv
Stage 2 val file: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_val_df.csv
File exists: True
File exists: True


In [3]:
if not STAGE2_TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Stage 2 train file not found. Run 2.1_Family_column.ipynb first: {STAGE2_TRAIN_PATH.resolve()}"
    )

if not STAGE2_VAL_PATH.exists():
    raise FileNotFoundError(
        f"Stage 2 val file not found. Run 2.1_Family_column.ipynb first: {STAGE2_VAL_PATH.resolve()}"
    )
stage2_train_df = pd.read_csv(STAGE2_TRAIN_PATH)
stage2_val_df = pd.read_csv(STAGE2_VAL_PATH)


print("Shape:", stage2_train_df.shape)



Shape: (5165, 57)


In [4]:
known_flags = [
    "storage_known",
    "ram_known",
    "condition_known",
]

for col in known_flags:
    print(f"{col}: {'EXISTS' if col in stage2_train_df.columns else 'MISSING'}")

for col in known_flags:
    print(f"{col}: {'EXISTS' if col in stage2_val_df.columns else 'MISSING'}")    

storage_known: MISSING
ram_known: MISSING
condition_known: MISSING
storage_known: MISSING
ram_known: MISSING
condition_known: MISSING


In [5]:
spec_cols = ["storage_gb", "ram_gb", "condition"]

display(
    pd.DataFrame({
        "dtype": stage2_train_df[spec_cols].dtypes,
        "missing_count": stage2_train_df[spec_cols].isna().sum(),
        "known_count": stage2_train_df[spec_cols].notna().sum(),
        "known_percentage": (
            stage2_train_df[spec_cols].notna().mean() * 100
        ).round(2),
    })
)

,dtype,missing_count,known_count,known_percentage
storage_gb,float64,4788,377,7.30
ram_gb,float64,5046,119,2.30
condition,object,4874,291,5.63


creating known flags  (storage,ram,condition)

In [6]:
stage2_train_df["storage_known"] = stage2_train_df["storage_gb"].notna()
stage2_train_df["ram_known"] = stage2_train_df["ram_gb"].notna()
stage2_train_df["condition_known"] = stage2_train_df["condition"].notna()

In [7]:
stage2_val_df["storage_known"] = stage2_val_df["storage_gb"].notna()
stage2_val_df["ram_known"] = stage2_val_df["ram_gb"].notna()
stage2_val_df["condition_known"] = stage2_val_df["condition"].notna()

In [8]:
known_flags = [
    "storage_known",
    "ram_known",
    "condition_known",
]

display(
    stage2_train_df[known_flags].agg(["sum", "count"]).T
)

display(
    stage2_train_df[known_flags].mean().mul(100).round(2)
    .to_frame("known_percentage")
)

,sum,count
storage_known,377,5165
ram_known,119,5165
condition_known,291,5165


,known_percentage
storage_known,7.30
ram_known,2.30
condition_known,5.63


In [9]:
print("Shape:", stage2_train_df.shape)
print("Unique ad IDs:", stage2_train_df["ad_id"].nunique())
print("Unique title groups:", stage2_train_df["title_group"].nunique())

Shape: (5165, 60)
Unique ad IDs: 5165
Unique title groups: 2240


In [10]:
display(stage2_train_df.dtypes.to_frame("dtype"))

,dtype
title,object
price,float64
location,object
ad_url,object
ad_id,int64
run_id,object
brand,object
storage_gb,float64
ram_gb,float64
storage_src,object


In [11]:
stage2_train_df['brand'].value_counts()

brand
apple       2462
samsung      749
xiaomi       426
oppo         361
honor        224
realme       178
infinix      138
vivo         128
huawei        99
motorola      26
zte           17
oneplus       17
tecno         15
google        13
itel          12
nokia          8
sony           7
lenovo         6
lg             3
Name: count, dtype: int64

In [12]:
stage2_train_df['model_key'].value_counts()

model_key
unknown                    465
apple iphone 13            372
apple iphone 17 pro max    342
apple iphone 11            242
apple iphone 16 pro max    147
                          ... 
huawei mate 20 pro           1
oppo a5x                     1
realme c4                    1
xiaomi redmi 5               1
oppo a60                     1
Name: count, Length: 723, dtype: int64

# Pipeline

------------------------------------------------


----------------------------------------------------------------------

# pipeline 

preparing categorical features + encodeing 

In [14]:

categorical_features = ["model_key", "brand", "family"]

known_features = [
    "storage_known",
    "ram_known",
    "condition_known",
]

feature_columns = categorical_features + known_features

# =========================
# Train
# =========================
train_df = stage2_train_df.loc[stage2_train_df["price"].notna()].copy()
X_train = train_df[feature_columns].copy()
y_train = train_df["price"].copy()
groups_train = train_df["title_group"].copy()

# =========================
# Validation
# =========================
val_df = stage2_val_df.loc[stage2_val_df["price"].notna()].copy()
X_val = val_df[feature_columns].copy()
y_val = val_df["price"].copy()
groups_val = val_df["title_group"].copy()

# =========================
# Check
# =========================
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

print("Missing train targets:", y_train.isna().sum())
print("Missing validation targets:", y_val.isna().sum())

print("Missing train groups:", groups_train.isna().sum())
print("Missing validation groups:", groups_val.isna().sum())

display(X_train.head())
display(X_val.head())


Train rows: 5165
Validation rows: 1252
X_train shape: (5165, 6)
X_val shape: (1252, 6)
Missing train targets: 0
Missing validation targets: 0
Missing train groups: 0
Missing validation groups: 0


,model_key,brand,family,storage_known,ram_known,condition_known
0,apple iphone 13,apple,iphone,False,False,False
1,honor magic v5,honor,magic,False,False,False
2,oppo find x9 pro,oppo,find,False,False,False
3,honor robot,honor,robot,True,True,False
4,xiaomi 17 ultra,xiaomi,xiaomi,False,False,False


,model_key,brand,family,storage_known,ram_known,condition_known
0,honor 500 pro,honor,honor numeric,False,False,False
1,apple iphone 13 pro max,apple,iphone,False,False,False
2,samsung s24 ultra,samsung,galaxy s,False,False,False
3,honor win rt,honor,win,False,False,False
4,apple iphone 13 pro max,apple,iphone,False,False,False


### Quick encoding check

This checks the encoded shape on the training rows. When an encoder is used in cross-validation, fit it on each training fold only.

In [15]:

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("known_flags", "passthrough", known_features)
])

print("Preprocessor is ready.")
print("Categorical features:", categorical_features)
print("Known flags:", known_features)



# Fit على Train فقط
X_train_transformed = preprocessor.fit_transform(X_train)
# Transform على Validation فقط
X_val_transformed = preprocessor.transform(X_val)

print("Original train shape:", X_train.shape)
print("Encoded train shape:", X_train_transformed.shape)

print("Original validation shape:", X_val.shape)
print("Encoded validation shape:", X_val_transformed.shape)

print("Encoding successful!")

Preprocessor is ready.
Categorical features: ['model_key', 'brand', 'family']
Known flags: ['storage_known', 'ram_known', 'condition_known']
Original train shape: (5165, 6)
Encoded train shape: (5165, 810)
Original validation shape: (1252, 6)
Encoded validation shape: (1252, 810)
Encoding successful!


pause
---------------------------

In [16]:
STAGE2_PATH.parent.mkdir(parents=True, exist_ok=True)
stage2_df.to_csv(STAGE2_PATH, index=False)
print("Updated Stage 2 file saved:", STAGE2_PATH.resolve())

Updated Stage 2 file saved: C:\Users\user\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_df.csv
